In [ ]:
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split

#load dataset from notebook 01
ds = load_dataset("Fllamber/TCGA-PANCAN-HiSeq-2770x20530")
ds.set_format(type='pandas')
df = ds['train'][:]

X = df.drop(columns=['cancer']) 
y = df['cancer']

#test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42)

#fitting and validation splits
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42)

In [ ]:
#standardization
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_fit)
X_fit_scaled = scaler.transform(X_fit)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [ ]:
#baseline logistic regression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, classification_report

baseline_logreg = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

baseline_logreg.fit(X_fit_scaled, y_fit)

y_val_pred_baselinelogreg = baseline_logreg.predict(X_val)

print("Accuracy_score:", accuracy_score(y_val, y_val_pred_baselinelogreg))
print("Baseline logistic regresiion macro-F1:", 
      f1_score(y_val, y_val_pred_baselinelogreg, average="macro"))
print("Classification report:")
print(classification_report(y_val, y_val_pred_baselinelogreg))

In [ ]:
#check best number of principal components
from sklearn.decomposition import PCA

variance_threshold = [0.80, 0.90, 0.95, 0.99]
results = []

for variance in variance_threshold:
    pca_test = PCA(n_components=variance)
    X_fit_pca_test = pca_test.fit_transform(X_fit_scaled)

    X_val_pca_test = pca_test.transform(X_val_scaled)

    model_log = LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    )
    model_log.fit(X_fit_pca_test, y_fit)

    y_val_pred_logreg = model_log.predict(X_val_pca_test)

    accuracy = accuracy_score(y_val, y_val_pred_logreg)
    f1_macro = f1_score(y_val, y_val_pred_logreg, average='macro')

    results.append({
        "Variance": variance,
        "Components": pca_test.n_components_,
        "Accuracy": accuracy,
        "Macro f1-score": f1_macro
    })

    print(f"PCA variance: {variance}")
    print(f"Number of components: {pca_test.n_components_}")
    print("Classification report:")
    print(classification_report(
        y_val, 
        y_val_pred_logreg))

results_df = pd.DataFrame(results)
print(results_df.round(3))

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.pipeline import Pipeline

def tune_model(name, estimator, param_distributions,
               X_fit, y_fit, X_val, y_val,
               reduction="none",
               n_iter=3):

    steps = []

    if reduction=="pca":
        steps.append(("scaler", StandardScaler()))
        steps.append(("pca", PCA(n_components=0.95)))

    elif reduction=="selection":
        steps.append(("remove_constant", VarianceThreshold(threshold=0.0)))
        steps.append(("scaler", StandardScaler()))
        steps.append(("select", SelectKBest(score_func=f_classif, k=1000)))

    elif reduction=="none":
        steps.append(("scaler", StandardScaler()))

    else:
        raise ValueError("reduction must be 'none, 'pca', or 'selection'")
    
    steps.append(("model", estimator))
    
    pipeline = Pipeline(steps)

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_distributions,
        n_iter=n_iter,
        scoring="f1_macro",
        cv=cv,
        random_state=42,
        n_jobs=1,
        verbose=1,
        refit=True,
        error_score="raise"
    )

    search.fit(X_fit, y_fit)

    best_model = search.best_estimator_
    y_val_pred_model = best_model.predict(X_val)

    accuracy_model = accuracy_score(y_val, y_val_pred_model)
    f1_macro_model = f1_score(y_val, y_val_pred_model, average='macro')

    print(f"{name}")
    print("Best parameters:", search.best_params_)
    print("Best CV Macro F1:", search.best_score_)
    print("Validation Accuracy:", accuracy_model)
    print("Validation Macro F1:", f1_macro_model)

    print("Validation classification report:")
    print(classification_report(y_val, y_val_pred_model))

    result_model = {
        "Model": name,
        "Feature reduction": reduction,
        "Best CV Macro F1": search.best_score_,
        "Validation Accuracy": accuracy_model,
        "Validation Macro F1": f1_macro_model,
        "Best Parameters": search.best_params_
    }

    return search, result_model

In [ ]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

models_to_tune = {
    "Random Forest": {
        "estimator": RandomForestClassifier(
            class_weight="balanced",
            random_state=42,
            n_jobs=1
        ),
        "params": {
            "model__n_estimators": [100, 200, 300],
            "model__max_depth": [10, 20, 30, None],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4, 8],
            "model__max_features": ["sqrt", 0.3, 0.5]
        }
    },

    "Extra Trees": {
        "estimator": ExtraTreesClassifier(
            class_weight="balanced",
            random_state=42,
            n_jobs=1
        ),
        "params": {
            "model__n_estimators": [100, 200, 300],
            "model__max_depth": [10, 20, 30, None],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4, 8],
            "model__max_features": ["sqrt", 0.3, 0.5]
        }
    },

    "LightGBM": {
        "estimator": LGBMClassifier(
            class_weight="balanced",
            random_state=42,
            verbosity=-1,
            n_jobs=1
        ),
        "params": {
            "model__n_estimators": [50, 100, 200, 300],
            "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
            "model__num_leaves": [15, 31, 50],
            "model__max_depth": [-1, 10, 20],
            "model__min_child_samples": [10, 20, 30, 50]
        }
    },

    "CatBoost": {
        "estimator": CatBoostClassifier(
            auto_class_weights="Balanced",
            random_seed=42,
            verbose=0,
            allow_writing_files=False,
            thread_count=1
        ),
        "params": {
            "model__iterations": [100, 200, 300, 500],
            "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
            "model__depth": [4, 5, 6, 8],
            "model__l2_leaf_reg": [1, 3, 5, 7]
        }
    },

    "XGBoost": {
        "estimator": XGBClassifier(
            objective="multi:softprob",
            eval_metric="mlogloss",
            tree_method="hist",
            random_state=42,
            n_jobs=1
        ),
        "params": {
            "model__n_estimators": [50, 100, 200, 300],
            "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
            "model__max_depth": [3, 4, 5, 7],
            "model__min_child_weight": [1, 3, 5],
            "model__subsample": [0.7, 0.8, 1.0],
            "model__colsample_bytree": [0.5, 0.7, 0.9]
        }
    }
}

In [ ]:
all_searches = {}
all_results = []

for name, config in models_to_tune.items():

    search, result = tune_model(
        name=name,
        estimator=config["estimator"],
        param_distributions=config["params"],
        X_fit=X_fit,
        y_fit=y_fit,
        X_val=X_val,
        y_val=y_val,
        reduction="selection",
        n_iter=3
    )

    all_searches[name] = search
    all_results.append(result)

all_results_df = pd.DataFrame(all_results)

best_results_df = all_results_df.sort_values(
    by="Validation Macro F1",
    ascending=False
).reset_index(drop=True)

print(best_results_df[
    [
        "Model",
        "Feature reduction",
        "Best CV Macro F1",
        "Validation Accuracy",
        "Validation Macro F1",
    ]
].round(4))

In [ ]:
different = rf_pred != et_pred

comparison = pd.DataFrame({
    "Actual cancer": y_val.iloc[different].values,
    "Random Forest prediction": rf_pred[different],
    "Extra Trees prediction": et_pred[different]
})

print(comparison)

In [ ]:
print(all_searches)

In [ ]:
final_model = all_searches["Random Forest"].best_estimator_

y_test_pred = final_model.predict(X_test)

print("Test accuracy:", accuracy_score(y_test, y_test_pred))
print(
    "Test macro F1:",
    f1_score(y_test, y_test_pred, average="macro")
)
print(classification_report(y_test, y_test_pred))

In [ ]:
import joblib

joblib.dump(final_model, "random_forest_cancer_model.joblib")
print("Model savec")

In [ ]:
import json

feature_names = X_fit.columns.tolist()

with open('feature_names.json', "w") as f:
    json.dump(feature_names, f)

print('Feature names saved')
len(feature_names)

In [ ]:
import mlflow
import mlflow.sklearn

# Store experiment results locally in mlruns folder
mlflow.set_tracking_uri("file:./mlruns")

# Create or select the experiment
mlflow.set_experiment("Cancer-Classification")

print("MLflow experiment creates")

In [ ]:
for model_name, search in all_searches.items():

    # Get best fitted model 
    best_model = search.best_estimator_

    # Calculate validation predictions
    val_predictions = best_model.predict(X_val)

    with mlflow.start_run(run_name=model_name):

        # Record the model best hyperparameters
        mlflow.log_params(search.best_params_)

        # Record the experiment results
        mlflow.log_metric(
            "cv_macro_f1",
            search.best_score_
        )

        mlflow.log_metric(
            "validation_accuracy",
            accuracy_score(y_val, val_predictions)
        )

        mlflow.log_metric(
            "validation_macro_f1",
            f1_score(
                y_val,
                val_predictions,
                average="macro"
            )
        )

        # Save fitted model as MLflow artifact
        mlflow.sklearn.log_model(
            sk_model=best_model,
            artifact_path="model"
        )

        mlflow.set_tag("model_name", model_name)
        mlflow.set_tag("dataset", "TCGA-PANCAN-HiSeq")

        print(f"Logged {model_name}")